In [0]:
%sql
-- Clientes con Compras Infladas (Tolerancia 2%)
SELECT 
    c.id_cliente,
    c.segmento_actividad,
    COUNT(f.id_transaccion) AS cantidad_compras_infladas,
    ROUND(AVG(f.desvio_pct), 2) AS desvio_promedio_abonado_pct
FROM platform_dev.gold_byma.fact_operaciones f
INNER JOIN platform_dev.gold_byma.dim_cliente c ON f.cliente_sk = c.cliente_sk
WHERE f.flag_compro_caro = TRUE
GROUP BY c.id_cliente, c.segmento_actividad
ORDER BY cantidad_compras_infladas DESC;

In [0]:
%sql
-- Top 10 Instrumentos con Mayor Desvío Promedio
SELECT 
    i.simbolo,
    i.descripcion,
    i.tipo_instrumento,
    COUNT(f.id_transaccion) AS operaciones_totales,
    ROUND(AVG(ABS(f.desvio_pct)), 2) AS desvio_promedio_absoluto_pct
FROM platform_dev.gold_byma.fact_operaciones f
LEFT JOIN platform_dev.gold_byma.dim_instrumento i ON f.instrumento_sk = i.instrumento_sk
GROUP BY i.simbolo, i.descripcion, i.tipo_instrumento
HAVING operaciones_totales > 5
ORDER BY desvio_promedio_absoluto_pct DESC
LIMIT 10;

In [0]:
%sql
-- Performance del Volumen Financiero por Canal (ARS vs USD)
SELECT 
    can.nombre_canal,
    can.tipo_canal,
    ROUND(SUM(CASE WHEN ins.moneda_base = 'ARS' THEN f.monto_total ELSE 0 END), 2) AS volumen_total_ars,
    ROUND(SUM(CASE WHEN ins.moneda_base = 'USD' THEN f.monto_total ELSE 0 END), 2) AS volumen_total_usd
FROM platform_dev.gold_byma.fact_operaciones f --
LEFT JOIN platform_dev.gold_byma.dim_canal can ON f.canal_sk = can.canal_sk --
LEFT JOIN platform_dev.gold_byma.dim_instrumento ins ON f.instrumento_sk = ins.instrumento_sk --
GROUP BY can.nombre_canal, can.tipo_canal
ORDER BY volumen_total_ars DESC;

In [0]:
%sql
-- Evolución Semanal de Proporción Compra/Venta (AL30 y AL30D)
SELECT 
    DATE_TRUNC('WEEK', fec.fecha) AS inicio_semana,
    ins.simbolo,
    COUNT(CASE WHEN f.tipo_operacion = 'Compra' THEN 1 END) AS cantidad_compras,
    COUNT(CASE WHEN f.tipo_operacion = 'Venta' THEN 1 END) AS cantidad_ventas,
    ROUND(
        COUNT(CASE WHEN f.tipo_operacion = 'Compra' THEN 1 END) / 
        NULLIF(COUNT(CASE WHEN f.tipo_operacion = 'Venta' THEN 1 END), 0), 2
    ) AS ratio_compra_venta
FROM platform_dev.gold_byma.fact_operaciones f
LEFT JOIN platform_dev.gold_byma.dim_instrumento ins ON f.instrumento_sk = ins.instrumento_sk
LEFT JOIN platform_dev.gold_byma.dim_fecha fec ON f.fecha_sk = fec.fecha_sk
WHERE ins.simbolo IN ('AL30', 'AL30D')
GROUP BY DATE_TRUNC('WEEK', fec.fecha), ins.simbolo
ORDER BY inicio_semana ASC, ins.simbolo;

In [0]:
%sql
-- Concentración de Operatoria de Clientes VIP
WITH TopClientes AS (
    SELECT f.cliente_sk, COUNT(*) AS transacciones_totales
    FROM platform_dev.gold_byma.fact_operaciones f
    GROUP BY f.cliente_sk
    ORDER BY transacciones_totales DESC
    LIMIT 5
),
PreferenciaActivos AS (
    SELECT 
        tc.cliente_sk,
        f.instrumento_sk,
        COUNT(*) AS operaciones_activo,
        DENSE_RANK() OVER (PARTITION BY tc.cliente_sk ORDER BY COUNT(*) DESC) AS ranking_preferencia
    FROM platform_dev.gold_byma.fact_operaciones f
    INNER JOIN TopClientes tc ON f.cliente_sk = tc.cliente_sk
    GROUP BY tc.cliente_sk, f.instrumento_sk
)
SELECT 
    c.id_cliente,
    c.segmento_actividad,
    i.simbolo AS activo_mas_operado,
    i.tipo_instrumento,
    pa.operaciones_activo AS cantidad_operaciones_en_este_activo
FROM PreferenciaActivos pa
INNER JOIN platform_dev.gold_byma.dim_cliente c ON pa.cliente_sk = c.cliente_sk
INNER JOIN platform_dev.gold_byma.dim_instrumento i ON pa.instrumento_sk = i.instrumento_sk
WHERE pa.ranking_preferencia = 1
ORDER BY pa.operaciones_activo DESC;

In [0]:
%sql
-- Preguntas de Valor de Negocio
SELECT 
    c.segmento_actividad,
    COUNT(f.id_transaccion) AS volumen_operaciones_totales,
    ROUND(SUM(f.monto_total), 2) AS dinero_total_movilizado,
    ROUND(AVG(f.monto_total), 2) AS ticket_promedio_por_operacion
FROM platform_dev.gold_byma.fact_operaciones f
INNER JOIN platform_dev.gold_byma.dim_cliente c ON f.cliente_sk = c.cliente_sk
GROUP BY c.segmento_actividad
ORDER BY ticket_promedio_por_operacion DESC;

In [0]:
%sql
-- Volumen Operado según el Día de la Semana
SELECT 
    fec.dia_semana,
    fec.es_fin_de_semana,
    COUNT(f.id_transaccion) AS cantidad_operaciones,
    ROUND(SUM(f.monto_total), 2) AS volumen_financiero_pesos
FROM platform_dev.gold_byma.fact_operaciones f
INNER JOIN platform_dev.gold_byma.dim_fecha fec ON f.fecha_sk = fec.fecha_sk
GROUP BY fec.dia_semana, fec.es_fin_de_semana
ORDER BY volumen_financiero_pesos DESC;